# Session 26 — CLIP: Connecting Vision & Language

**Computer Vision & AI Course · Practical Notebook**

In this session we build and use **CLIP** (Contrastive Language-Image Pre-training) — the model that puts images and text into one shared embedding space.

### What you will do
- **Part A — Warm-up:** compute similarity between vectors by hand, so the core operation of CLIP feels natural.
- **Part B — CLIP from scratch:** train a small CLIP on **FLICKR-8K** (8,000 images × 5 captions) and search images with a text query.
- **Part C — Pre-trained OpenAI CLIP:** load ViT-B/32 and do zero-shot classification and semantic search — no training at all.

> Part B follows `CLIP_from_scratch.ipynb` from *Modern Computer Vision with PyTorch* (2nd ed.), Chapter 16 — repo: https://bit.ly/mcvp-2e
> Part C follows `OpenAI_CLIP.ipynb` from the same chapter.

**Requirements:** GPU runtime recommended (Colab works). Part B needs a **Kaggle account** to download FLICKR-8K.

---
## Part A — Warm-up: Embeddings & Similarity

CLIP's entire inference recipe is: *turn things into vectors, compare with a dot product*.
Before trusting that with 40,000 image–text pairs, let's do it with vectors small enough to read.

In [ ]:
import numpy as np

# Pretend embeddings in a tiny 4-d space.
# In real CLIP these would come out of the image / text encoders (256-d in Part B).
image_embeddings = {
    "photo_of_dog":   np.array([0.9, 0.1, 0.0, 0.2]),
    "photo_of_cat":   np.array([0.1, 0.9, 0.1, 0.1]),
    "photo_of_plane": np.array([0.0, 0.1, 0.9, 0.3]),
}
text_embeddings = {
    "a dog":   np.array([0.8, 0.2, 0.1, 0.1]),
    "a cat":   np.array([0.2, 0.8, 0.0, 0.2]),
    "a plane": np.array([0.1, 0.0, 0.8, 0.4]),
}

In [ ]:
def cosine_similarity(a, b):
    """Dot product of L2-normalised vectors -> value in [-1, 1]."""
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return float(a @ b)

# Score every image against every text -> the N x N similarity matrix from the slides
print(f"{'':16s}" + "".join(f"{t:>10s}" for t in text_embeddings))
for img_name, img_vec in image_embeddings.items():
    row = [cosine_similarity(img_vec, txt_vec) for txt_vec in text_embeddings.values()]
    print(f"{img_name:16s}" + "".join(f"{v:10.3f}" for v in row))

Notice the **diagonal** holds the highest values — each image is most similar to its matching text.
That is exactly what CLIP's contrastive training enforces, just in 256+ dimensions with real photos and captions.

To turn similarities into a probability over labels, we apply a **softmax** (optionally sharpened by a *temperature*):

In [ ]:
def softmax(x, temperature=1.0):
    x = np.array(x) / temperature
    e = np.exp(x - x.max())          # subtract max for numerical stability
    return e / e.sum()

sims = [cosine_similarity(image_embeddings["photo_of_dog"], t) for t in text_embeddings.values()]
for temp in [1.0, 0.1]:
    probs = softmax(sims, temperature=temp)
    print(f"temperature={temp}: ", {name: round(p, 3) for name, p in zip(text_embeddings, probs)})

**Takeaway:** a lower temperature makes the distribution *sharper* (more confident).
CLIP learns its temperature during training. Keep this in mind — you'll see `temperature` again in the loss in Part B.

---
## Part B — Building CLIP From Scratch on FLICKR-8K

We now assemble a real CLIP: **ResNet-50** as the image encoder, **DistilBERT** as the text encoder,
and a **projection head** on each side mapping into a shared 256-d space.

Dataset: **FLICKR-8K** — 8,000 images, each with 5 human-written captions (~1 GB, via Kaggle).

> ⚠️ **Before running:** you need a Kaggle account and API key (`kaggle.json`), and you must accept the
> dataset's terms at https://www.kaggle.com/datasets/adityajn105/flickr8k — otherwise the download will fail.

### B1. Install the required packages and clone the chapter repo

In [ ]:
%%capture
import os
if not os.path.exists("MCVP2e-CLIP"):
    !git clone https://github.com/sizhky/MCVP2e-CLIP.git
    %pip install -r MCVP2e-CLIP/requirements.txt
%cd MCVP2e-CLIP

### B2. Import the required packages

Everything model-related lives in the repo's `clip` package — we will open up the key classes as we go.

In [ ]:
import itertools
import pandas as pd
from torch_snippets import *
from clip.core import download_flickr8k_from_kaggle
from clip.config import ClipConfig
from clip.dataset import CLIPDataset
from clip.models import CLIP

### B3. Provide your Kaggle credentials

Replace the `XXXX` placeholders with your own username and key (Kaggle → Account → *Create New API Token*).

In [ ]:
%%writefile kaggle.json
{"username":"XXXX","key":"XXXX"}

### B4. Download the FLICKR-8K dataset

We also add an `id` column so each of the 5 captions of an image shares the same image id —
that keeps image–caption pairs aligned.

In [ ]:
kaggle_json_path = P("kaggle.json")
data_download_path = P("/content/flickr-8k-kaggle/")

download_flickr8k_from_kaggle(kaggle_json_path, data_download_path)

df = pd.read_csv(data_download_path / "captions.txt")
df["id"] = [id_ for id_ in range(len(df) // 5) for _ in range(5)]
df.to_csv(data_download_path / "captions.csv", index=None)
df.head()

### B5. Set up the training configuration

`ClipConfig` holds every experiment knob: the learning rates of the image and text encoders, epochs, backbone
names, image/text embedding sizes, max text length, and the projection dimension. For now we only point it at
the data and train **1 epoch**.

In [ ]:
config = ClipConfig()
config.image_path = data_download_path / "Images"
config.captions_csv_path = data_download_path / "captions.csv"
config.debug = False            # True -> tiny subset, useful for a smoke test
config.epochs = 1
config.save_and_logging_steps = 50
config

### B6. Create the training and validation datasets

Key ideas inside `CLIPDataset` (from `clip/dataset.py`):

```python
class CLIPDataset(Dataset):
    def __init__(self, df, config, mode):
        self.tokenizer = DistilBertTokenizer.from_pretrained(config.distilbert_text_tokenizer)
        self.image_filenames = df.image.tolist()
        self.captions = df.caption.tolist()
        self.encoded_captions = self.tokenizer(          # ALL captions tokenized up front
            self.captions, padding=True, truncation=True,
            max_length=config.max_length)
        self.transforms = get_transforms(config)         # resize to fixed size + normalise

    def __getitem__(self, idx):
        item = {key: torch.tensor(values[idx])
                for key, values in self.encoded_captions.items()}
        image = read(f"{self.config.image_path}/{self.image_filenames[idx]}")
        item["image"] = torch.tensor(self.transforms(image=image)["image"]).permute(2, 0, 1)
        item["caption"] = self.captions[idx]
        return item
```

Every item = tokenized caption (`input_ids`, `attention_mask`) **plus** the transformed image — one ready-made pair.

In [ ]:
trn_ds, val_ds = CLIPDataset.train_test_split(config)
len(trn_ds), len(val_ds)

### B7. Load the model

In [ ]:
model = CLIP(config).to(config.device)

### B8. The key components of the model

**i. `ImageEncoder`** — a pre-trained ResNet-50 (via `timm`), classifier head removed, giving a **2048-d** vector per image:

```python
class ImageEncoder(nn.Module):
    def __init__(self, model_name=CFG.model_name, pretrained=CFG.pretrained, trainable=CFG.trainable):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained,
                                       num_classes=0, global_pool="avg")
        for p in self.model.parameters():
            p.requires_grad = trainable
    def forward(self, x):
        return self.model(x)
```

**ii. `TextEncoder`** — DistilBERT; we keep the hidden state of the **CLS token** (index 0) as the sentence embedding (**768-d**):

```python
class TextEncoder(nn.Module):
    def __init__(self, ...):
        super().__init__()
        self.model = DistilBertModel.from_pretrained(...)
        self.target_token_idx = 0            # CLS token
    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = output.last_hidden_state
        return last_hidden_state[:, self.target_token_idx, :]
```

**iii. `ProjectionHead`** — 2048-d (image) and 768-d (text) can't be compared, so each side is projected into the **same 256-d space**:

```python
class ProjectionHead(nn.Module):
    def __init__(self, embedding_dim, projection_dim=CFG.projection_dim, dropout=CFG.dropout):
        super().__init__()
        self.projection = nn.Linear(embedding_dim, projection_dim)
        self.gelu = nn.GELU()
        self.fc = nn.Linear(projection_dim, projection_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(projection_dim)
    def forward(self, x):
        projected = self.projection(x)
        x = self.gelu(projected)
        x = self.fc(x)
        x = self.dropout(x)
        x = x + projected                     # residual connection
        x = self.layer_norm(x)
        return x
```

### B9. The `CLIPModel` forward pass — where the loss lives

```python
def forward(self, batch):
    # 1. encode both modalities
    image_features = self.image_encoder(batch["image"])
    text_features  = self.text_encoder(input_ids=batch["input_ids"],
                                       attention_mask=batch["attention_mask"])
    # 2. project into the shared space (same dimensionality)
    image_embeddings = self.image_projection(image_features)
    text_embeddings  = self.text_projection(text_features)

    # 3. the N x N similarity matrix, scaled by temperature
    logits = (text_embeddings @ image_embeddings.T) / self.temperature

    # 4. soft targets from intra-modal similarity
    images_similarity = image_embeddings @ image_embeddings.T
    texts_similarity  = text_embeddings  @ text_embeddings.T
    targets = F.softmax((images_similarity + texts_similarity) / 2 * self.temperature, dim=-1)

    # 5. symmetric cross-entropy: texts->images AND images->texts
    texts_loss  = cross_entropy(logits,   targets,   reduction='none')
    images_loss = cross_entropy(logits.T, targets.T, reduction='none')
    loss = (images_loss + texts_loss) / 2.0
    return {"loss": loss.mean()}          # dict -> Hugging Face Trainer compatible
```

Why **soft targets** instead of the identity matrix? Two nearly identical photos in a batch shouldn't be
forced to have zero similarity with each other's captions — the intra-modal similarities acknowledge that.

### B10. Optimizer — three parameter groups, three learning rates

The pre-trained encoders get gentle updates; the freshly initialised projection heads learn faster.

In [ ]:
import torch

params = [
    {"params": model.image_encoder.parameters(), "lr": config.image_encoder_lr},
    {"params": model.text_encoder.parameters(),  "lr": config.text_encoder_lr},
    {"params": itertools.chain(
        model.image_projection.parameters(),
        model.text_projection.parameters()),
     "lr": config.head_lr, "weight_decay": config.weight_decay},
]
optimizer = torch.optim.AdamW(params, weight_decay=0.0)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=config.patience, factor=config.factor)

### B11. Train with the Hugging Face `Trainer`

The Trainer wraps `opt.zero_grad()`, `loss.backward()`, evaluation and checkpointing —
the same pattern you saw with the detectron trainer and the mmaction runner.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    evaluation_strategy="steps",
    logging_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    learning_rate=config.head_lr,
    logging_steps=config.save_and_logging_steps,
    save_steps=config.save_and_logging_steps,
    eval_steps=config.save_and_logging_steps,
    logging_dir="./logs",
    metric_for_best_model="loss",
    label_names=["image", "input_ids"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=trn_ds,
    eval_dataset=val_ds,
    optimizers=(optimizer, lr_scheduler),
)
trainer.train()

### B12. Fetch embeddings for all validation images

Once trained, we run **every** validation image through the image encoder + projection head *once*, and keep
the resulting embedding bank for fast search.

In [ ]:
from clip.core import make_train_valid_dfs, build_loaders
from clip.models import CLIPModel
from transformers import DistilBertTokenizer
from tqdm import tqdm

CFG = config

def get_image_embeddings(valid_df, model_path):
    tokenizer = DistilBertTokenizer.from_pretrained(CFG.distilbert_text_tokenizer)
    valid_loader = build_loaders(valid_df, tokenizer, mode="valid")

    model = CLIPModel().to(CFG.device)
    model.load_state_dict(torch.load(model_path, map_location=CFG.device))
    model.eval()

    valid_image_embeddings = []
    with torch.no_grad():
        for batch in tqdm(valid_loader):
            image_features = model.image_encoder(batch["image"].to(CFG.device))
            image_embeddings = model.image_projection(image_features)
            valid_image_embeddings.append(image_embeddings)
    return model, torch.cat(valid_image_embeddings)

_, valid_df = make_train_valid_dfs()
model, image_embeddings = get_image_embeddings(valid_df, "best.pt")

### B13. Find matches for a text query

The search recipe:
1. tokenize + encode the **query text** → project to the shared space,
2. **L2-normalise** both the text embedding and the image embedding bank,
3. one matrix multiply scores the query against *every* image,
4. `torch.topk` picks the best `n`.

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2

def find_matches(model, image_embeddings, query, image_filenames, n=9):
    tokenizer = DistilBertTokenizer.from_pretrained(CFG.distilbert_text_tokenizer)
    encoded_query = tokenizer([query])
    batch = {key: torch.tensor(values).to(CFG.device)
             for key, values in encoded_query.items()}
    with torch.no_grad():
        text_features = model.text_encoder(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        text_embeddings = model.text_projection(text_features)

    image_embeddings_n = F.normalize(image_embeddings, p=2, dim=-1)
    text_embeddings_n  = F.normalize(text_embeddings,  p=2, dim=-1)
    dot_similarity = text_embeddings_n @ image_embeddings_n.T

    values, indices = torch.topk(dot_similarity.squeeze(0), n * 5)
    matches = [image_filenames[idx] for idx in indices[::5]]

    _, axes = plt.subplots(3, 3, figsize=(10, 10))
    for match, ax in zip(matches, axes.flatten()):
        image = cv2.imread(f"{CFG.image_path}/{match}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)
        ax.axis("off")
    plt.show()

find_matches(model,
             image_embeddings,
             query="dogs on the grass",
             image_filenames=valid_df["image"].values,
             n=9)

**What just happened?** Nowhere in FLICKR-8K is there a label called *"dogs on the grass"* — yet the model
returns a grid of dogs on grass. It matched the **meaning** of your sentence to image content. This is semantic search.

**Try it:** change the query — `"children playing in water"`, `"a man riding a bicycle"` — and re-run.

---
## Part C — Zero-Shot with Pre-trained OpenAI CLIP

Training CLIP from scratch takes considerable compute and data. **OpenAI CLIP** was trained on
**400 million** image–text pairs — so instead of training, we just load it.

### C1. Install the required packages

In [ ]:
%%capture
%pip install ftfy regex tqdm
%pip install git+https://github.com/openai/CLIP.git

### C2. Load the pre-trained model

`clip.load` returns the model **and** its matching image `preprocess` transform (resize / crop / normalise —
it must match what the model saw during training).

In [ ]:
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print(f"Loaded ViT-B/32 on {device} — "
      f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

### C3. Zero-shot classification of a single image

We grab the CLIP paper's own architecture figure and ask the model what it is. The candidate labels are
**anything we want** — that's the point of zero-shot.

In [ ]:
# The figure from the OpenAI CLIP repository (a diagram)
!wget -q https://raw.githubusercontent.com/openai/CLIP/main/CLIP.png -O CLIP.png

image = preprocess(Image.open("CLIP.png")).unsqueeze(0).to(device)
text = clip.tokenize(["a diagram", "a dog", "a cat"]).to(device)

with torch.no_grad():
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:", probs)   # expected ≈ [[0.9927  0.0042  0.0030]]

The model has never been *trained* to classify "diagram vs dog vs cat" — it simply measures which text
embedding lies closest to the image embedding. Change the label list and you have a brand-new classifier,
with zero retraining.

### C4. Zero-shot with prompt engineering

Wrapping labels in a sentence like *"a photo of a {object}"* usually beats the bare word — the model saw
captions during training, not single words. Let's classify a real photo (bundled with scikit-image, no
download needed).

In [ ]:
from skimage import data as skdata
import matplotlib.pyplot as plt

cat_image = Image.fromarray(skdata.chelsea())     # a photo of a cat
plt.imshow(cat_image); plt.axis("off"); plt.show()

labels = ["cat", "dog", "horse", "car", "airplane"]
prompts = [f"a photo of a {label}" for label in labels]

image_input = preprocess(cat_image).unsqueeze(0).to(device)
text_input = clip.tokenize(prompts).to(device)

with torch.no_grad():
    logits_per_image, _ = model(image_input, text_input)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]

for label, p in sorted(zip(labels, probs), key=lambda x: -x[1]):
    print(f"{label:10s} {p:.4f}")

### C5. Mini semantic search — your own `find_matches`, powered by OpenAI CLIP

Same recipe as Part B (encode → normalise → dot product → top-k), but with the pre-trained encoders.
We use four bundled sample images as our tiny "image library".

In [ ]:
import numpy as np

# a tiny image library (all bundled with scikit-image)
library = {
    "cat":       Image.fromarray(skdata.chelsea()),
    "astronaut": Image.fromarray(skdata.astronaut()),
    "coffee":    Image.fromarray(skdata.coffee()),
    "motorcycle": Image.fromarray(skdata.stereo_motorcycle()[0]),
}

# 1. embed every image once (the "offline" step)
with torch.no_grad():
    image_batch = torch.cat([preprocess(im).unsqueeze(0) for im in library.values()]).to(device)
    image_features = model.encode_image(image_batch)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

def search(query, n=1):
    """Return the library images most similar to a text query."""
    with torch.no_grad():
        text_features = model.encode_text(clip.tokenize([query]).to(device))
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        similarity = (text_features @ image_features.T).squeeze(0)
    best = similarity.argsort(descending=True)[:n]
    names = list(library.keys())
    return [(names[i], float(similarity[i])) for i in best]

for query in ["a cup of espresso on a table",
              "a person in a space suit",
              "a furry pet sleeping",
              "a fast two-wheeled vehicle"]:
    print(f"{query!r:40s} -> {search(query)}")

Each free-form sentence found the right image — no query mentions "coffee", "astronaut", "cat" or
"motorcycle" by name. The shared embedding space is doing all the work.

### C6. Probing the limits (discussion)

CLIP is powerful but not magic. Things worth knowing:

- **Counting & spatial reasoning are weak** — "three dogs left of a cat" is hard for it.
- **Typographic attacks** — a real apple with a paper note saying "iPod" can be classified as *iPod*: the text
  in the image leaks into the embedding.
- **Prompt wording matters** — "a photo of a {label}" vs "{label}" can shift accuracy by several percent;
  the paper uses ensembles of 80 prompts.
- **It inherits web biases** — its training data is 400M pairs scraped from the internet.

**Where you have already used it:** the text encoder inside **Stable Diffusion** (Session 25) is a CLIP text
encoder — the 77×768 prompt embedding that steers denoising comes from exactly the model you used today.

---
## Exercises (homework)

1. **Part B:** re-run `find_matches` with 3 queries of your own. Find one query that *fails* — why do you
   think it fails? (Hint: think about what FLICKR-8K photos contain.)
2. **Part C:** build a zero-shot classifier for 5 categories of your choice and test it on 3 images from the
   web. Compare bare labels vs `"a photo of a ..."` prompts.
3. **Stretch:** in `search()`, replace the library with ~20 of your own photos and build a personal semantic
   photo search.
4. **Think:** why does CLIP training not need any human annotators, while ImageNet training did? What does
   this imply about how far the approach can scale?

## Summary

| Concept | One-liner |
|---|---|
| Contrastive pre-training | pull matched (image, text) pairs together, push mismatched apart |
| Projection head | maps 2048-d (image) / 768-d (text) into one shared 256-d space |
| Temperature | sharpens/softens the softmax over similarities |
| Symmetric loss | cross-entropy over rows (images→texts) *and* columns (texts→images) |
| Zero-shot | classify with any label list via "a photo of a {object}" — no retraining |
| Semantic search | normalise embeddings, dot product, top-k |

**Next session:** from labels to masks — open-vocabulary segmentation (SAM).